# 00. Dataset Metadata — GSE287331

This notebook extracts and standardizes the phenotypic metadata for dataset GSE287331.  
It retrieves sample-level information from GEO, harmonizes tissue annotations into a 3-class labeling scheme (normal, adjacent/benign, tumor), and stores the final table as a compressed Parquet file for reproducible downstream integration.

**Source: GEO accession GSE287331, platform Illumina Infinium MethylationEPIC v1.0 BeadChip**


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║   POLITECNICO DI TORINO — MSc Mathematical Engineering           ║
# ║   THESIS: Advanced Study of Epigenetic Mechanisms in Neoplasms   ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Author:       Elisabetta Roviera (s328422)                       ║
# ║ Supervisors:  Dr. Sandro Gambino, Prof. Alfredo Benso            ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Notebook:     00-dataset-metadata-GSE287331                      ║
# ║ Description: : Extracts and standardizes phenotypic metadata,    ║
# ║                ensuring consistent sample labeling and           ║
# ║                reproducible downstream integration.              ║
# ║ Dataset(s):   GSE287331                                          ║
# ║ Repository:   github.com/elisabettaroviera/THESIS                ║
# ╟──────────────────────────────────────────────────────────────────╢
# ║ Date: 10-Nov-2025 | Python 3.11.13                               ║
# ╚══════════════════════════════════════════════════════════════════╝


## Libraries

In [ ]:
!pip install -q GEOparse polars pyarrow lz4


In [ ]:
import os
import re
from pathlib import Path
import GEOparse
import polars as pl


## 1. Builf pheno_GSE287331

In [ ]:
# ============================================
# Build pheno.parquet for GSE287331 using GEOparse + Polars
# Columns:
#   - geo_accession
#   - sample_name
#   - tissue_type_raw
#   - label_3class (0 = normal, 1 = adjacent/case-benign, 2 = tumor)
#   - idat_basename
# ============================================

import os
from pathlib import Path
import GEOparse
import polars as pl

# -------------------------
# CONFIG
# -------------------------
GSE_ID        = "GSE287331"  # <--- GEO accession, NOT the Parquet path!
DESTDIR       = "/kaggle/working/geoparse_cache"       # where to cache SOFT files
OUT_PHENO_PAR = "/kaggle/working/pheno_gse287331.parquet"

Path(DESTDIR).mkdir(parents=True, exist_ok=True)

# (Optional) if you also want to remember where the beta Parquet lives:
BETA_PARQUET_PATH = "/kaggle/input/gse287331-lz4-parquet/GSE287331_lz4.parquet"

# -------------------------
# Helper: map tissue -> 3-class label
# -------------------------
def map_tissue_to_label(tissue: str | None) -> int | None:
    if tissue is None:
        return None
    t = tissue.strip().upper()
    # Normal controls
    if t == "HDB":
        return 0
    # Case-benign / adjacent axis
    if t in {"CUB", "OQ", "AN"}:
        return 1
    # Tumor
    if t in {"TU", "TUMOR"}:
        return 2
    # Anything unexpected -> None (to be inspected manually)
    return None

# -------------------------
# 1) Download/load GSE with GEOparse
# -------------------------
gse = GEOparse.get_GEO(
    geo=GSE_ID,
    destdir=DESTDIR,
    annotate_gpl=False,  # we don't need probe annotation here
    how="full",          # full SOFT
    silent=True,
)

# -------------------------
# 2) Extract per-sample metadata into a list of dicts
# -------------------------
rows: list[dict] = []

for gsm_name, gsm in gse.gsms.items():
    meta = gsm.metadata

    # geo_accession (GSM ID)
    geo_accession = gsm.get_accession()  # e.g. "GSM8744013"

    # sample_name from title
    sample_name = meta.get("title", [geo_accession])[0]

    # tissue_type_raw from characteristics_ch1 like "tissue: HDB"
    tissue_raw = None
    for val in meta.get("characteristics_ch1", []):
        if val is None:
            continue
        # look for "tissue: XXX"
        if "tissue:" in val.lower():
            tissue_raw = val.split(":", 1)[1].strip()
            break

    # idat_basename from description (e.g. "202234810048_R01C01")
    desc_list = meta.get("description", [])
    idat_basename = desc_list[0].strip() if desc_list else None

    # Map to 3-class label
    label_3class = map_tissue_to_label(tissue_raw)

    rows.append(
        {
            "geo_accession": geo_accession,
            "sample_name": sample_name,
            "tissue_type_raw": tissue_raw,
            "label_3class": label_3class,
            "idat_basename": idat_basename,
        }
    )

# -------------------------
# 3) Build Polars DataFrame and save to Parquet (LZ4)
# -------------------------
pheno = pl.DataFrame(rows)

# Cast label_3class to small integer (Int8) to save space
pheno = pheno.with_columns(
    pl.col("label_3class").cast(pl.Int8)
)

print(pheno.head())
print(pheno["tissue_type_raw"].value_counts())
print(pheno["label_3class"].value_counts())

pheno.write_parquet(
    OUT_PHENO_PAR,
    compression="lz4",
    statistics=True,
)

print(f"✅ Saved phenotype table to: {OUT_PHENO_PAR}")
print(f"Shape: {pheno.shape}")
